# 12.3: Memberi Tangan pada LLM

Model bahasa hanya bisa menghasilkan teks. Ia tidak bisa menghitung dengan
tepat, dan tidak tahu apa pun yang terjadi setelah data latihnya dibekukan.

Tool calling menutup kedua lubang itu. Anda memberi model sebuah daftar fungsi
Python, model memutuskan mana yang perlu dipanggil, dan **kode Anda** yang
menjalankannya.

Bedanya dengan Sesi 11 hanya satu argumen tambahan. Sungguh.

In [ ]:
import json
import os
import sys
import urllib.request

sys.path.insert(0, "..")

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"
print("siap")

## 1. Kita coba ajukan dua pertanyaan berbeda

Kita akan menyuruh LLM untuk menghitung dan memberikan prakiraan cuaca

In [ ]:
from google.genai import types
TANPA_THINKING = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0)
)

for soal in ["Berapa hasil 1247 dikali 89?",
             "Berapa suhu di Jakarta sekarang?"]:
    resp = client.models.generate_content(
        model=MODEL, contents=soal, config=TANPA_THINKING
    )
    print(f"T: {soal}")
    print(f"J: {resp.text.strip()}")
    print()

print("hitungan sebenarnya:", 1247 * 89)

## 2. Tool pertama, sebuah kalkulator

Sebuah tool adalah fungsi Python biasa. Tidak ada kelas khusus, tidak ada
dekorator, tidak ada pendaftaran.

Dua hal yang dibaca model dari fungsi Anda:

- **docstring** menjadi deskripsi, yang dipakai model untuk memutuskan kapan
  fungsi ini relevan,
- **type hint** menjadi schema, yang dipakai model untuk menyusun argumennya.

Keduanya bukan hiasan. Docstring yang malas menghasilkan tool yang jarang
dipanggil, atau dipanggil di saat yang salah.

Satu catatan praktis: jaga agar parameternya tetap sederhana, cukup `str`,
`int`, atau `float`. Parameter berbentuk list atau dict sering sampai dalam
keadaan tidak rapi dan menyulitkan pemula.

In [ ]:
def kalkulator(a: float, b: float, operator: str) -> str:
    # TODO 1: tulis docstring yang menjelaskan fungsi ini dan ketiga
    # argumennya. Docstring inilah yang dibaca model untuk memutuskan
    # kapan tool ini relevan, jadi tulis dengan jelas.
    if operator == "/" and b == 0:
        return "tidak terdefinisi, pembagi nol"
    hasil = {"+": a + b, "-": a - b, "*": a * b,
             "/": a / b if b else 0.0}[operator]
    return str(hasil)


print(kalkulator(1247, 89, "*"))

Sekarang serahkan fungsi itu kepada model. Inilah satu argumen yang menjadi
seluruh pelajaran hari ini:

```python
config=types.GenerateContentConfig(tools=[kalkulator])
```

SDK akan mengubah fungsi Anda menjadi schema, menjalankan fungsinya ketika
model memintanya, lalu mengirimkan hasilnya kembali ke model. Semua itu terjadi
di dalam satu pemanggilan.

In [ ]:
resp = client.models.generate_content(
    model=MODEL,
    contents="Berapa hasil 1247 dikali 89?",
    config=types.GenerateContentConfig(tools=[kalkulator]),
)
print(resp.text.strip())

## 3. Tool kedua, data dari luar

Kalkulator menyelesaikan masalah hitungan. Masalah kedua adalah model tidak tahu
apa pun tentang hari ini.

Kita pakai Open-Meteo, yang tidak memerlukan API key maupun pendaftaran. Ia
meminta koordinat, bukan nama kota, jadi tool kita membawa kamus kecil. Ini
sekaligus menunjukkan bahwa sebuah tool boleh melakukan apa saja yang bisa
dilakukan Python, termasuk mencari di tabel.

Perhatikan `try` dan `except`. Sebuah tool yang gagal sebaiknya mengembalikan
kalimat, bukan melempar error. Kalau ia melempar, seluruh alur berhenti. Kalau
ia mengembalikan kalimat, model bisa menjelaskan keadaannya kepada pengguna.

In [ ]:
KOTA = {
    "Jakarta": (-6.21, 106.85),
    "Bandung": (-6.91, 107.61),
    "Surabaya": (-7.25, 112.75),
    "Medan": (3.59, 98.67),
    "Makassar": (-5.15, 119.43),
}


def cuaca(kota: str) -> str:
    """Ambil suhu udara terkini sebuah kota di Indonesia.

    Args:
        kota: nama kota, misalnya "Jakarta" atau "Bandung"
    """
    if kota not in KOTA:
        return f"Kota {kota} tidak ada dalam daftar: {', '.join(KOTA)}"
    lat, lon = KOTA[kota]
    url = (f"https://api.open-meteo.com/v1/forecast?latitude={lat}"
           f"&longitude={lon}&current=temperature_2m")
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            data = json.loads(r.read())
        suhu = data["current"]["temperature_2m"]
        return f"Suhu di {kota} saat ini {suhu} derajat Celcius."
    except Exception as e:
        return f"Gagal mengambil data cuaca: {e}"


print(cuaca("Jakarta"))

## 4. Dua tool, model yang memilih sendiri

Satu tool mengajarkan mekanismenya. Dua tool mengajarkan konsep sebenarnya:
model **memilih**.

Jalankan tiga pertanyaan berikut. Tidak satu pun dari kita yang memberi tahu
model tool mana yang harus dipakai.

In [ ]:
# TODO 3: susun satu GenerateContentConfig yang membawa KEDUA tool,
# lalu jalankan ketiga pertanyaan di bawah ini dengan config itu.
ALAT = None

for soal in ["Berapa 15 persen dari 50 juta?",
             "Bagaimana cuaca di Bandung hari ini?",
             "Halo, kamu bisa bantu apa saja?"]:
    resp = client.models.generate_content(model=MODEL, contents=soal,
                                          config=ALAT)
    print(f"T: {soal}")
    print(f"J: {resp.text.strip()[:220]}")
    print()

Pertanyaan ketiga tidak memanggil tool sama sekali, karena memang tidak perlu.
Itu bagian dari keputusan yang diambil model, dan ia akan menjadi dasar
intent detection di 12.4.

## 5. Melihat mekanismenya secara langsung

Kita akan lihat apa yang sebenarnya terjadi di belakang layar saat kode melakukan tool calling

In [ ]:
manual = types.GenerateContentConfig(
    tools=[kalkulator],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

resp = client.models.generate_content(
    model=MODEL, contents="Berapa 1247 dikali 89?", config=manual)

panggilan = resp.candidates[0].content.parts[0].function_call
print("nama fungsi :", panggilan.name)
print("argumen     :", dict(panggilan.args))

# TODO 4: jalankan fungsi kalkulator memakai argumen yang diminta model,
# lalu cetak hasilnya. Inilah langkah yang tadi dikerjakan SDK diam-diam.
hasil = None
print("hasil lokal :", hasil)


Tiga langkah, selalu berurutan, dan tidak pernah berubah:

1. Model **meminta**, dengan menyebut nama fungsi dan argumennya.
2. **Kode Anda** menjalankan fungsi itu.
3. Model **membaca** hasilnya dan menyusun jawaban akhir.

Model tidak pernah menjalankan kode apa pun. Ia hanya meminta. Kalau Anda tidak
menjalankan permintaannya, tidak ada yang terjadi.


In [ ]:
balasan = types.Part.from_function_response(
    name=panggilan.name, response={"hasil": hasil})

akhir = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Content(role="user", parts=[types.Part(text="Berapa 1247 dikali 89?")]),
        resp.candidates[0].content,
        types.Content(role="assistant", parts=[balasan]),
    ],
    config=manual,
)
print("jawaban akhir:", akhir.text)



## 6. Di LangChain, hal yang sama dengan antarmuka yang berbeda

Kita memakai SDK Gemini secara langsung karena itu yang sudah Anda kenal sejak
Sesi 11, dan karena kodenya paling pendek.

Di kebanyakan codebase, Anda akan menemukan bentuk LangChain. Konsepnya persis
sama, hanya namanya berbeda:

```python
from langchain_core.tools import tool

@tool
def kalkulator(a: float, b: float, operator: str) -> str:
    """Hitung operasi aritmatika dua angka."""
    ...

llm_dengan_tool = llm.bind_tools([kalkulator])
pesan = llm_dengan_tool.invoke("Berapa 1247 dikali 89?")
print(pesan.tool_calls)
```

Keuntungannya, kode yang sama bisa dipindah ke model lain tanpa banyak berubah.
Biayanya, ada empat istilah baru sebelum baris pertama berjalan.

Sel di atas hanya untuk dibaca, tidak untuk dijalankan hari ini.

## 7. Giliran Anda

Tulis satu tool ketiga dari pekerjaan Anda sendiri, lalu tambahkan ke daftar dan
uji dengan pertanyaan yang seharusnya memanggilnya.

Ide yang mudah dikerjakan sekarang: menghitung usia dari tahun lahir, mengubah
mata uang dengan kurs tetap, atau menghitung cicilan dari pokok dan bunga.

In [ ]:
# TODO 5: tulis satu tool ketiga milik Anda sendiri, lengkap dengan docstring
# dan type hint. Tambahkan ke daftar tools, lalu uji dengan pertanyaan yang
# seharusnya memanggilnya.
def tool_saya():
    pass


TOOLS = None
resp = None
print(resp.text.strip())